# Prerequisites

In [19]:
pip install requests openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Making Your First API Call in OpenRouter

## Your first request and setup

In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

response = client.chat.completions.create(
    model="x-ai/grok-4.1-fast:free",  # Your desired model
    messages=[
        {"role": "user", "content": "Write a haiku about debugging code at 2 AM"}
    ]
)

print(response.choices[0].message.content)

Screen glows at two a.m.,  
Elusive bug mocks tired eyes,  
Dawn's light brings commit.


## Troubleshoot environmet

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

print("ENV VALUE:", os.getenv("OPENROUTER_API_KEY"))


ENV VALUE: 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


In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="sk-or-v1-ISI_KEYMU_DI_SINI"
)

print("Client OK")


Client OK


In [4]:
key = os.getenv("OPENROUTER_API_KEY")
print("Key is None?", key is None)
print("Key length:", len(key) if key else None)
print("Key raw repr:", repr(key))


Key is None? False
Key length: 73
Key raw repr: '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'


In [5]:
print(os.getenv("OPENROUTER_API_KEY"))


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


In [6]:
import requests
import os

key = os.getenv("OPENROUTER_API_KEY")

headers = {
    "Authorization": f"Bearer {key}",
}

resp = requests.get("https://openrouter.ai/api/v1/auth/key", headers=headers)
print(resp.json())


{'data': {'label': 'sk-or-v1-fe0...313', 'is_provisioning_key': False, 'limit': None, 'limit_reset': None, 'limit_remaining': None, 'include_byok_in_limit': False, 'usage': 0, 'usage_daily': 0, 'usage_weekly': 0, 'usage_monthly': 0, 'byok_usage': 0, 'byok_usage_daily': 0, 'byok_usage_weekly': 0, 'byok_usage_monthly': 0, 'is_free_tier': True, 'expires_at': None, 'rate_limit': {'requests': -1, 'interval': '10s', 'note': 'This field is deprecated and safe to ignore.'}}}


# Model Routing For Resilience

## Setting up manual fallbacks

In [18]:
response = client.chat.completions.create(
   model="kwaipilot/kat-coder-pro:free",  # Primary choice
   messages=[
       {"role": "user", "content": "Explain quantum computing in simple terms"}
   ],
   extra_body={
       "models": ["nvidia/nemotron-nano-12b-v2-vl:free", "alibaba/tongyi-deepresearch-30b-a3b:free"]
   }   
)

print(f"Response from: {response.model}")
print(response.choices[0].message.content)

Response from: kwaipilot/kat-coder-pro:free
Sure! Let's break it down in simple terms:

**Quantum computing** is a type of computing that uses the strange rules of quantum physics to process information in ways that regular computers can't.

### First, let's talk about regular computers:
- Regular computers use **bits** as the smallest unit of data.
- A bit can be either a **0** or a **1**.
- Everything a computer does—text, images, videos—is ultimately made up of lots of 0s and 1s.

### Now, quantum computers are different:
- Instead of bits, they use **qubits** (quantum bits).
- A qubit can be **0**, **1**, or **both at the same time**! This is called **superposition**.

### Let’s use an analogy:
Imagine a light switch:
- In a regular computer, the switch is either **off (0)** or **on (1)**.
- In a quantum computer, the switch can be **off**, **on**, or **both off and on at the same time**—like a magical switch that exists in two states simultaneously until you check it.

### Another

In [ ]:
response = client.chat.completions.create(
   model="openrouter/auto:free",
   messages=[
       {"role": "user", "content": "Debug this Python code in 3 sentences: def factorial(n): return n * factorial(n-1)"}
   ]
)

print(f"Auto router selected: {response.model}")
print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found for openrouter/auto:free.', 'code': 404}, 'user_id': 'user_35m838U6dWNGRQvdSUPIPbn6vLA'}

## Building effective fallback strategies

pakai backup dari provider yang berbeda

In [24]:
# Good fallback chain: different providers, decreasing cost
response = client.chat.completions.create(
   model="alibaba/tongyi-deepresearch-30b-a3b:free",
   messages=[
       {"role": "user", "content": "Your prompt here"}
   ],
   extra_body={
       "models": [
           "x-ai/grok-4.1-fast:free",                  # Close performance
           "meituan/longcat-flash-chat:free",           # Cheaper
           "kwaipilot/kat-coder-pro:free"     # Free backup
       ]
   }   
)

## Finding models for your fallback chain

In [27]:
def get_provider_models(api_key: str, provider: str) -> list[str]:
   r = requests.get(
       "https://openrouter.ai/api/v1/models",
       headers={"Authorization": f"Bearer {api_key}"}
   )
   return [m["id"] for m in r.json()["data"] if m["id"].startswith(provider)]

# Build fallbacks across providers
openai_models = get_provider_models('OPENROUTER_API_KEY', "openai/")
anthropic_models = get_provider_models('OPENROUTER_API_KEY', "anthropic/")

## Streaming For Real-time Responses

### Basic streaming setup

In [29]:
response = client.chat.completions.create(
   model="x-ai/grok-4.1-fast:free",
   messages=[
       {"role": "user", "content": "Write a detailed explanation of how neural networks learn"}
   ],
   stream=True
)

for chunk in response:
   if chunk.choices[0].delta.content is not None:
       print(chunk.choices[0].delta.content, end="")

# How Neural Networks Learn: A Detailed Explanation

Neural networks, the backbone of modern artificial intelligence and machine learning, "learn" by iteratively adjusting their internal parameters to make better predictions or decisions based on data. This process mimics biological learning in a highly mathematical way but is fundamentally an optimization problem. Below, I'll break it down step-by-step, from the basics to advanced concepts, using intuitive explanations, analogies, and key math where relevant.

## 1. The Structure of a Neural Network
A neural network consists of interconnected **nodes** (or "neurons") organized in **layers**:
- **Input Layer**: Takes raw data (e.g., pixel values of an image).
- **Hidden Layers**: Process the data through weighted connections.
- **Output Layer**: Produces predictions (e.g., class probabilities for image classification).

Each connection between neurons has a **weight** (a number representing importance) and each neuron has a **bias** (a

## Building a better streaming handler

In [31]:
def stream_response(model, messages, show_progress=True):
   response = client.chat.completions.create(
       model=model,
       messages=messages,
       stream=True
   )
  
   complete_response = ""
  
   for chunk in response:
       if chunk.choices[0].delta.content is not None:
           content = chunk.choices[0].delta.content
           complete_response += content
          
           if show_progress:
               print(content, end="", flush=True)
  
   if show_progress:
       print()  # Add final newline
  
   return complete_response

# Use it with different models
result = stream_response(
   "x-ai/grok-4.1-fast:free",
   [{"role": "user", "content": "Explain quantum entanglement like I'm 12 years old"}]
)

Imagine you and your best friend each get a magic box with a coin inside. The coins look normal, but they're *special*—they're **entangled**, like invisible twins connected by quantum magic.

Here's how it works:

1. **You separate them far apart.** You take your box to school, your friend takes theirs to the moon (way farther than that!).

2. **You open your box first.** Let's say your coin lands on **heads**. Boom! At the *exact same second*, no matter how far away, your friend's coin instantly becomes **tails**. Not "maybe" tails—it *is* tails. It's like they decided together the moment you looked.

3. **It's instant and spooky.** There's no phone call, no signal traveling between them. Albert Einstein called it "spooky action at a distance" because it seems faster than light, which breaks normal rules. But quantum physics says it's real!

Why does this happen? Tiny particles (like electrons or photons) can get linked in a "quantum dance." Until you check one, they're in a fuzzy "bo

# Handling Reasoning Tokens In OpenRouter

## Get to know reasoning tokens

Reasoning tokens = token yang dipakai model untuk berpikir internal

reasoning tokent itu kayak explainable code
Output tokens = token yang muncul sebagai jawaban akhirnya

In [33]:
response = client.chat.completions.create(
   model="openai/gpt-oss-20b:free",
   messages=[
       {"role": "user", "content": "How many 'r's are in the word 'strrawberry'?"}
   ],
   max_tokens=2048,
   extra_body={
       "reasoning": {
           "max_tokens": 512
       }
   }
)

print("Final answer:")
print(response.choices[0].message.content)
print("\nReasoning process:")
print(response.choices[0].message.reasoning)

Final answer:
There are **four** 'r' letters in “strrawberry”.

Reasoning process:
User asks: "How many 'r's are in the word 'strrawberry'?" We need to identify the number of 'r' characters in that string. Let's count: 's t r r a w b e r r y'? Wait the spelling is 'strrawberry' - note the double 'r' at start? Actually 'strrawberry' spelled s t r r a w b e r r y. Let's write indices: 1:s,2:t,3:r,4:r,5:a,6:w,7:b,8:e,9:r,10:r,11:y. So r's at 3,4,9,10: total 4. So answer: 4. Maybe rhetorical? It's a trick. Yes answer: 4.


## Controlling reasoning intensity

### High effort
- Model memakai ≈80% dari max_tokens untuk internal reasoning.
- Sisanya dipakai untuk output (jawaban yang kamu lihat).
Contoh:
max_tokens = 4000
80% = 3200 token dipakai untuk berpikir
20% = 800 token untuk jawaban final

### Medium effort
Model memakai ≈50% dari max_tokens untuk reasoning internal.

### Low effort
Model memakai ≈20% dari max_tokens untuk reasoning.

In [38]:
# High effort reasoning for complex problems
response = client.chat.completions.create(
   model="qwen/qwen3-coder:free",
   messages=[
       {"role": "user", "content": "Solve this step by step: If a train travels 240 miles in 3 hours, then speeds up by 20 mph for the next 2 hours, how far does it travel total?"}
   ],
   max_tokens=4000,  # High effort will use ~3200 tokens for reasoning
   extra_body={
       "reasoning": {
           "effort": "high" 
       }
   }
)

print("Problem solution:")
print(response.choices[0].message.content)
print("\nStep-by-step reasoning:")
print(response.choices[0].message.reasoning)

Problem solution:
I need to find the total distance traveled by calculating the distance for each part of the journey.

**Step 1: Find the initial speed**
- Distance = 240 miles
- Time = 3 hours
- Speed = Distance ÷ Time = 240 miles ÷ 3 hours = 80 mph

**Step 2: Find the new speed after speeding up**
- Initial speed = 80 mph
- Speed increase = 20 mph
- New speed = 80 mph + 20 mph = 100 mph

**Step 3: Find the distance traveled in the second part**
- New speed = 100 mph
- Time = 2 hours
- Distance = Speed × Time = 100 mph × 2 hours = 200 miles

**Step 4: Calculate the total distance**
- First part distance = 240 miles
- Second part distance = 200 miles
- Total distance = 240 miles + 200 miles = 440 miles

Therefore, the train travels a total distance of **440 miles**.

Step-by-step reasoning:
None


## Preserving reasoning in conversations

code di bawah ini mmenyimpan memori percakapan

In [41]:
# First message with reasoning
response = client.chat.completions.create(
   model="openai/gpt-oss-20b:free",
   messages=[
       {"role": "user", "content": "Should I invest in renewable energy stocks? Consider both risks and opportunities."}
   ],
   extra_body={
       "reasoning": {
           "max_tokens": 3000
       }
   }
)

# Build conversation history with reasoning preserved
messages = [
   {"role": "user", "content": "Should I invest in renewable energy stocks? Consider both risks and opportunities."},
   {
       "role": "assistant",
       "content": response.choices[0].message.content,
       "reasoning_details": response.choices[0].message.reasoning_details  # Preserve reasoning
   },
   {"role": "user", "content": "What about solar energy specifically? How does that change your analysis?"}
]

# Continue conversation with reasoning context
follow_up = client.chat.completions.create(
   model="openai/gpt-oss-20b:free",
   messages=messages,
   extra_body={
       "reasoning": {
           "max_tokens": 2000
       }
   }
)

print("Follow-up answer:")
print(follow_up.choices[0].message.content)
print("\nContinued reasoning:")
print(follow_up.choices[0].message.reasoning)

Follow-up answer:
## Solar Energy: Opportunity & Risk Lens

Below is a sector‑by‑sector dissection that focuses specifically on **photovoltaic (PV) solar**—the technology that powers the majority of new renewable capacity worldwide.  
I’ve kept the discussion high‑level so you can quickly map it onto your own portfolio strategy, but feel free to ask for deeper dives on any sub‑topic.

| **Dimension** | **Solar‑Specific Opportunity** | **Solar‑Specific Risk** |
|---------------|--------------------------------|-------------------------|
| **Technology** | Rapid efficiency gains (Q‑Line, 25 %+ certified cells) and durability (15–25 yr warranties). | Technological brittleness: a new cell architecture could undercut current prices. |
| **Cost** | Panel module costs fell ~ 70 % from 2010 to 2023; manufacturing capacity now > 200 GW. | Supply‑chain bottlenecks (silicon, polysilicon, cell‑sub‑components) can spike costs mid‑cycle. |
| **Regulation/Policy** | U.S. 30 % ITC (Industrial Technolo

# Working With Multimodal Models in OpenRouter

## Understanding multimodal capabilities

## Working with images

In [47]:
response = client.chat.completions.create(
   model="tngtech/deepseek-r1t-chimera:free",
   messages=[
       {
           "role": "user",
           "content": "What's happening in this image? Describe the scene in detail."
       }
   ],
   extra_body={
       "attachments": [
           {
               "type": "image/jpeg",
               "url": "https://www.google.com/url?sa=i&url=https%3A%2F%2Fid.pinterest.com%2Fpin%2F579908889556186472%2F&psig=AOvVaw277sMmFwmY5bgHFspiNVlQ&ust=1763798161545000&source=images&cd=vfe&opi=89978449&ved=0CBIQjRxqFwoTCJi57KjigpEDFQAAAAAdAAAAABAT"
           }
       ]
   }
)

print(response.choices[0].message.content)

Alright, so I need to describe what's happening in this image in detail. The problem is, I don't have the image in front of me. Hmm. Maybe the user expects me to ask for the image or clarify that I can't see it. But they might also be testing if I can handle such a situation or if I can provide a generic response.

Wait, maybe the image is part of a previous conversation or context that's missing here. Let me check. No, there's no prior context given. So, if I were a person trying to answer this, I'd first acknowledge that I can't see the image and ask for more details or a description.

But the user hasn't provided any specifics, so I can't just make up a scene. That would be misleading. The best approach is to politely explain the limitation and request more information. Alternatively, if I assume they might be referring to a common or hypothetical image, I could describe a generic scene, but that's risky because it might not match what they're thinking of.

Perhaps they're using thi

In [49]:
import base64

def encode_image_to_base64(image_path):
   with open(image_path, "rb") as image_file:
       encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
   return encoded_string

# Analyze a local screenshot
encoded_image = encode_image_to_base64("image.jpg")

response = client.chat.completions.create(
   model="tngtech/deepseek-r1t-chimera:free",
   messages=[
       {
           "role": "user",
           "content": "This is a screenshot of a data dashboard. What insights can you extract from the charts and metrics shown?"
       }
   ],
   extra_body={
       "attachments": [
           {
               "type": "image/jpg",
               "data": encoded_image
           }
       ]
   }
)

print(response.choices[0].message.content)

Okay, so I'm looking at this data dashboard, and I need to figure out what insights I can extract from it. Since I don't have the actual screenshot, I'll have to think about typical elements that might be present in a dashboard and how to analyze them. Let's start by imagining what a standard data dashboard might include.

First, dashboards usually have multiple charts and metrics. Common ones could be line charts, bar graphs, pie charts, and key performance indicators (KPIs) displayed as numbers. Let me break this down step by step.

**1. Identifying the Components:**
   - **Time Series Chart (Line or Bar Chart):** This might show trends over time, like monthly sales, user sign-ups, or website traffic.
   - **Pie or Donut Chart:** Probably showing distribution, such as market share, product categories, or customer demographics.
   - **KPI Metrics:** These are usually big numbers at the top, like total revenue, active users, conversion rate, etc.
   - **Geographical Map:** If there's a

## Processing PDF documents

In [51]:
def encode_pdf_to_base64(pdf_path):
   with open(pdf_path, "rb") as pdf_file:
       encoded_string = base64.b64encode(pdf_file.read()).decode('utf-8')
   return encoded_string

# Analyze a research paper
encoded_pdf = encode_pdf_to_base64("test.pdf")

response = client.chat.completions.create(
   model="tngtech/deepseek-r1t-chimera:free",
   messages=[
       {
           "role": "user",
           "content": "What is the content of the file?"
       }
   ],
   extra_body={
       "attachments": [
           {
               "type": "application/pdf",
               "data": encoded_pdf
           }
       ]
   }
)

print(response.choices[0].message.content)

Okay, the user is asking, "What is the content of the file?" but they haven't provided any specific file or context. Let me break this down.

First, I need to understand that the question is too vague. Without knowing which file they're referring to, I can't possibly provide the content. It could be any file on any system, and I don't have access to external files unless they're shared in the conversation.

I should consider possible scenarios. Maybe the user forgot to attach a file or mention the file name. Alternatively, they might be referring to a file in a previous part of the conversation that I'm not recalling. But looking back, there's no mention of any file earlier.

Next, I need to respond in a helpful way. Since I don't have enough information, the best approach is to ask for clarification. I should prompt the user to provide more details about the file they're asking about, such as the file name, format, or context.

Also, I should keep my response friendly and clear, makin

# Using Structured Outputs

## Anatomy of structured output requests

In [ ]:
"response_format": {
   "type": "json_schema",           # Always this for structured outputs
   "json_schema": {
       "name": "your_schema_name",  # Name for your schema
       "strict": True,              # Enforce strict compliance
       "schema": {
           # Your actual JSON schema definition goes here
       }
   }
}

## Sentiment analysis example

In [53]:
response = client.chat.completions.create(
   model="tngtech/deepseek-r1t-chimera:free",
   messages=[
       {"role": "user", "content": "Analyze the sentiment: 'This movie was amazing!'"}
   ],
   extra_body={
       "response_format": {
           "type": "json_schema",
           "json_schema": {
               "name": "sentiment_analysis",
               "strict": True,
               "schema": {
                   "type": "object",
                   "properties": {
                       "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                       "confidence": {"type": "number"}
                   },
                   "required": ["sentiment", "confidence"]
               }
           }
       }
   }
)

import json
result = json.loads(response.choices[0].message.content)
print(result)

{'sentiment': 'positive', 'confidence': 0.98}


Here’s what’s happening in this schema:

- sentiment: A string field restricted to three specific values using enum. The model can't return anything outside of "positive", "negative", or "neutral"
- confidence: A number field for the model's confidence score
- required: Both fields must be present in the response - the model can't skip them
- strict: True: Enforces rigid compliance with the schema structure